In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

import gpjax as gpx
import optax as ox
import paramax as px

from non_parametric_pro import ula

from non_parametric_pro.density import ProParameters, pro_logdensity_fn, pro_score_fn
from non_parametric_pro.ula import parametric_ula
from non_parametric_pro.util import prediction_basis, nlpd_gp, nlpd_pro
from non_parametric_pro.data.uci import load_uci_regression_dataset
from non_parametric_pro.adaptation.parameter_adaptation import parameter_adaptation
from non_parametric_pro.inducing import PointInducingBasis, compute_inducing_basis
from non_parametric_pro.gp import _full_gp_basis

from blackjax.util import run_inference_algorithm

from sklearn.preprocessing import StandardScaler

jax.config.update("jax_enable_x64", True)

In [2]:
key = jr.PRNGKey(0)

example = load_uci_regression_dataset("servo", split=1)

In [3]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_train = scaler_x.fit_transform(example.x_train)
y_train = scaler_y.fit_transform(example.y_train)

x_test = scaler_x.transform(example.x_test)
y_test = scaler_y.transform(example.y_test)

In [78]:
D = x_train.shape[1]

data = gpx.Dataset(X=x_train, y=y_train)
kernel = gpx.kernels.Matern32(lengthscale=jnp.ones((D,)))
prior = gpx.gps.Prior(mean_function=gpx.mean_functions.Zero(), kernel=kernel)
likelihood = gpx.likelihoods.Gaussian(num_datapoints=data.n)
posterior = prior * likelihood

optim = ox.adam(learning_rate=0.01)

gp_posterior, _ = gpx.fit(
    model=posterior,
    objective=lambda p, d: -gpx.objectives.conjugate_mll(p, d),
    train_data=data,
    optim=optim,
    num_iters=500,
    verbose=False,
)

gp_latent_dist = gp_posterior.predict(x_test, train_data=data)
gp_predictive_dist = gp_posterior.likelihood(gp_latent_dist)

gp_predictive_mean = gp_predictive_dist.mean
gp_predictive_std = jnp.sqrt(gp_predictive_dist.variance)

gp_sigma = gp_posterior.likelihood.obs_stddev
gp_kernel = gp_posterior.prior.kernel

print("GP NLPD:", nlpd_gp(y_test, gp_predictive_mean, gp_predictive_std))

GP NLPD: 0.39627018458935803


In [79]:
inducing = False

if inducing:
    M = 120
    key, subkey = jr.split(key)
    inducing_idx = jr.choice(subkey, x_train.shape[0], shape=(M,), replace=False)
    inducing_basis = PointInducingBasis(x_train[inducing_idx])

    basis, residual_std = compute_inducing_basis(inducing_basis, gp_kernel, x_train)

else:
    M = x_train.shape[0]
    basis, residual_std = _full_gp_basis(gp_posterior, x_train), None

sigma = gpx.parameters.SigmoidBounded(px.unwrap(gp_sigma), low=0.1, high=100.0)

pro_params = ProParameters(
    y=y_train,
    basis=basis,
    step_size=1e-3,
    sigma=sigma,
    alpha=1.0,
    residual_std=residual_std
)

J = 30

initial_position = jr.normal(key, (M, J))

In [80]:
adapt = False

if adapt:

    adaptation = parameter_adaptation(
        ula,
        pro_logdensity_fn,
        pro_params,
        x_train=x_train,
        initial_kernel=gp_kernel,
        warmup_steps=1000,
        sigma_adapt_every=10,
        kernel_adapt_every=100,
        objective_fn=pro_score_fn,
        inducing_basis=inducing_basis if inducing else None,
        sigma_optimizer=ox.adam(1e-1),
        kernel_optimizer=ox.adam(1e-2),
        progress_bar=True
    ) 

    adaptation_results, adaptation_info = adaptation.run(key, initial_position, num_steps=5000)
    adaptation_kernel = jax.tree.map(lambda x: x[-1], adaptation_info.kernel)
    adaptation_params = adaptation_results.parameters
    adaptation_position = adaptation_results.state.position

    fig, ax = plt.subplots(1, 4, figsize=(10, 2.5))

    ax[0].plot(px.unwrap(adaptation_info.sigma))
    ax[1].plot(px.unwrap(adaptation_info.kernel.lengthscale))
    ax[2].plot(px.unwrap(adaptation_info.kernel.variance))
    ax[3].plot(-adaptation_info.sampler_info.score[10:])
    plt.tight_layout()
    plt.show()

In [83]:
algorithm = parametric_ula(pro_logdensity_fn, adaptation_params if adapt else pro_params)

out, info = run_inference_algorithm(
    rng_key=key,
    inference_algorithm=algorithm,
    num_steps=2000,
    initial_position=adaptation_position if adapt else initial_position,
    progress_bar=True
)

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:00&lt;?]</div>

In [84]:
test_basis, test_cov = prediction_basis(gp_kernel, x_train, x_test, pro_params, inducing_basis=inducing_basis if inducing else None)

nlpd_pro(y_test, test_basis, test_cov, out.position, sigma=pro_params.sigma)

Array(0.26744373, dtype=float64)